# Embeddings Explorations
## Have you ever wondered how LLM understand text?

We humans are smart, so we just read more and more and ask for reflections to improve our reading skills.

It is quite similar for LLMs. When we are training modern LLMs, it is called `self-supervised` learning that model update parameters automatically during training. And fun fact, GPT-3.5's success is highly contributed to human feedback, just as 'reflection' in real life!

## But I mean, how do we visualize how LLM understand text?

We know vectors, a quantity with both magnitude and direction -- Isn't that perfect for the semantic meaning for text?

Let me clarify, we know dot product. In LLMs, if two tokens have a dot product that is close to 1, the semantic meaning is really close (imagine in a diagram where two vector with very small angle in between), if close to 0, the semantic meaning is really far apart.

## 1. Creating token embeddings

Let's illustrate how the token ID to embedding vector conversion works with a hands-on
example. Suppose we have the following four input tokens with IDs 2, 3, 5, and 1:

In [1]:
import torch
input_ids = torch.tensor([2, 3, 5, 1])

For the sake of simplicity and illustration purposes, suppose we have a small vocabulary of
only 6 words (instead of the 50,257 words in the BPE tokenizer vocabulary), and we want
to create embeddings of size 3 (in GPT-3, the embedding size is 12,288 dimensions):

The more dimensions per token, the richer semantic meaning it will get.

In `PyTorch` it is incredibly simple to create a token embedding, just give your `vocabulary size` and your `dimension number` using `torch.nn.Embedding`. 

In [2]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123) # Random seed
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

Let's see the underlying weight matrix of this embedding layer!

In [3]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


We can see that the weight matrix of the embedding layer contains small, random values.
- The **vertical** column is every dimenion of the token
- The **horizontal** row is each token

Let's obtain the embedding layer correspoinding to a random token:

In [4]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


Hmmm, right corresponding to the 4th token in the matrix!

Ohhh, so..
<div class='alert alert-block alert-success'>
This is essentially a look-up
operation that retrieves rows from the embedding layer's weight matrix via a token ID.
</div>

Let's apply that to all four input IDs we defined earlier (`torch.tensor([2,3,5,1])`)

In [5]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


Corresponding to number 3, 4, 6, 2 tokens!

## 2. Positional Embeddings (Encoding word positions)

Now let's create a layer with 50257 vocabulary size (GPT2) and 256 dimensions (less than GPT2)

In [6]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

Instantiating the dataloader (Exactly the same in `Data Loader Explorations.ipynb`).

In [7]:
from torch.utils.data import Dataset, DataLoader
import tiktoken


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [10]:
max_length = 4
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=1, shuffle=False
)

inputs, targets = next(iter(dataloader))

In [11]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])

Inputs shape:
 torch.Size([8, 4])


As we can see, the token ID tensor is 8x4-dimensional, meaning that the data batch
consists of 8 text samples (Rows) with 4 tokens each (Columns).

Let's now use the `embedding layer` to embed these token IDs into 256 dimensional vectors:

In [12]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


- 8 batches
- 4 tokens per batch
- 256 dimensions per token

**Perfect.**

Now comes to the big part:

For a GPT model's absolute embedding approach, we just need to create another
`embedding layer` that has the same dimension as the token_embedding_layer:

In [ ]:
context_length = max_length # Just 4 so little☠️
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [15]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


`arange` basically generates a 1-dimensional tensor containing values within a specified range, with a defined step size. It is commonly used for creating sequences of numbers.

TL;DR: Same to use as `for i in range(min, max, step)`

In this case`[0, 1, 2, ..., max_length-1]`

OK enough explaining let's be 
## **serious**
Let's add token embeddings and positoinal embeddings together!

In [16]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


Great! This forms `context vectors`
which
 can actually be the component being processed by the main LLM modules!
